# Data Extraction: Customer Lifetime Value Dataset

This notebook extracts customer features from Google BigQuery's [TheLook E-Commerce](https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce) public dataset for building a Customer Lifetime Value (CLV) model using the BG/NBD + Gamma-Gamma probabilistic framework.

**Pipeline:**
1. Connect to BigQuery with cost controls
2. Execute parameterized CLV feature engineering SQL
3. Validate data quality (BG/NBD assumptions)
4. Export to CSV for modeling in `02_clv_bgnbd.ipynb`

In [ ]:
import pandas as pd
import os
from google.cloud import bigquery
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Config
PROJECT_ID = "churn-portfolio-project"
SQL_FILE_PATH = "../sql/clv_features.sql"
DATA_PATH = "../data/raw/clv_data.csv"

# Parameters
CUTOFF_DATE = "2025-10-01"       # End of holdout window (simulated "today")
CALIBRATION_END = "2025-04-04"   # End of calibration period (cutoff - 180 days)

## CLV Modeling Framework

We use a **temporal train/holdout split** to validate the model before deployment:

| Period | Dates | Purpose |
|--------|-------|---------|
| Calibration | Before 2025-04-04 | Fit BG/NBD + Gamma-Gamma models |
| Holdout | 2025-04-04 → 2025-10-01 (180 days) | Validate CLV predictions |

**BG/NBD Inputs (computed at calibration end):**

| Feature | Definition | Notes |
|---------|------------|-------|
| `frequency` | Repeat purchases (total_orders - 1) | 0 = one-time buyer (~88% of customers) |
| `recency` | Days from first to last purchase | 0 for one-time buyers |
| `T` | Customer age in days at calibration end | Must be > 0 |
| `monetary_value` | Avg revenue per repeat transaction | Fallback to AOV for one-time buyers |

**Design Decision:** One-time buyers are NOT excluded — BG/NBD handles them natively and their low CLV output is the correct, informative answer.

## Feature Engineering

Features are extracted from three source tables:

### BG/NBD Core Inputs
*Source: `order_items` (calibration period)*

| Feature | Description |
|---------|-------------|
| `frequency` | Number of repeat purchases (total_orders - 1) |
| `recency` | Days from first to last purchase within calibration |
| `T` | Customer age at calibration end (days since first purchase) |
| `monetary_value` | Avg order value on repeat transactions |
| `avg_order_value` | Overall avg order value (context + fallback) |

### Holdout Labels
*Source: `order_items` (holdout period)*

| Feature | Description |
|---------|-------------|
| `actual_holdout_transactions` | Orders placed in holdout window |
| `actual_holdout_revenue` | Revenue generated in holdout window |

### Demographic Features
*Source: `users`*

| Feature | Description |
|---------|-------------|
| `customer_tenure_days` | Days since account creation |
| `age`, `gender` | Demographics |
| `traffic_source` | Acquisition channel |
| `country` | Geographic market |

### Engagement Features
*Source: `events`*

| Feature | Description |
|---------|-------------|
| `total_sessions` | Site visit count |
| `days_since_last_visit` | Browsing recency |
| `cart_events`, `product_view_events` | Purchase intent signals |

In [ ]:
# Load and display SQL query
with open(SQL_FILE_PATH) as f:
    query = f.read()

# Show first 50 lines for reference
print("\n".join(query.split("\n")[:50]))
print("\n... [truncated] ...")

## Query Execution

In [ ]:
# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Set up query job configuration with parameters
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=500 * 1024**2,  # 500MB safety cap
    query_parameters=[
        bigquery.ScalarQueryParameter("cutoff_date",      "DATE", CUTOFF_DATE),
        bigquery.ScalarQueryParameter("calibration_end",  "DATE", CALIBRATION_END),
    ],
)

In [ ]:
# Dry run: Estimate cost before execution
job_config.dry_run = True
job = client.query(query, job_config=job_config)

bytes_processed = job.total_bytes_processed
mb = bytes_processed / 1e6
cost = bytes_processed / 1e12 * 5  # $5 per TB

print(f"Estimated data scan: {mb:.2f} MB")
print(f"Estimated cost: ${cost:.4f}")

In [ ]:
# Execute query (or load cached data)
if os.path.exists(DATA_PATH):
    print(f"Loading cached data from {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    if mb > 500:
        raise ValueError(f"Query too large ({mb:.0f} MB). Adjust parameters.")

    print("Executing BigQuery...")
    job_config.dry_run = False
    df = client.query(query, job_config=job_config).to_dataframe()

    # Cache locally
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Saved to {DATA_PATH}")

print(f"\nDataset shape: {df.shape[0]:,} customers × {df.shape[1]} features")

## Data Validation

In [ ]:
# Schema check
print("=== Schema ===")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() > 0 else "No missing values")

In [ ]:
# BG/NBD data quality assertions
assert df['user_id'].is_unique,                       "Duplicate user IDs found"
assert (df['frequency'] >= 0).all(),                  "frequency must be non-negative"
assert (df['recency'] <= df['T']).all(),               "recency cannot exceed T"
assert (df['T'] > 0).all(),                            "T must be positive (T > 7 filter)"
assert (df['monetary_value'] > 0).all(),               "monetary_value must be positive for Gamma-Gamma"
assert df['frequency'].dtype in ['int64', 'Int64'],    "frequency should be integer"

print("✓ All BG/NBD data quality checks passed")

In [ ]:
# One-time buyer rate (key BG/NBD insight)
one_time_rate = (df['frequency'] == 0).mean()
print(f"=== Purchase Frequency Distribution ===")
print(f"One-time buyers (frequency=0): {(df['frequency']==0).sum():,} ({one_time_rate:.1%})")
print(f"Repeat buyers (frequency≥1):   {(df['frequency']>=1).sum():,} ({1-one_time_rate:.1%})")
print(f"\nFrequency stats:")
print(df['frequency'].describe())

In [ ]:
# BG/NBD input summary
bgnbd_cols = ['frequency', 'recency', 'T', 'monetary_value']
print("=== BG/NBD Core Inputs ===")
df[bgnbd_cols].describe().round(2)

In [ ]:
# Holdout label summary
holdout_buyers = (df['actual_holdout_transactions'] > 0).sum()
print(f"=== Holdout Period (180 days) ===")
print(f"Customers who purchased in holdout: {holdout_buyers:,} ({holdout_buyers/len(df):.1%})")
print(f"Total holdout transactions:         {df['actual_holdout_transactions'].sum():,}")
print(f"Total holdout revenue:              ${df['actual_holdout_revenue'].sum():,.2f}")

# Categorical feature summary
categorical_cols = ['gender', 'traffic_source', 'country']
for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts().head(5))

In [ ]:
# Sample records
df.head()

---

**Next:** [02_clv_bgnbd.ipynb](./02_clv_bgnbd.ipynb) — BG/NBD + Gamma-Gamma model fitting and CLV computation